In [31]:
from pathlib import Path
import pandas as pd
import os

BASE_DIR = Path.cwd().parent
KOL_PATH = BASE_DIR / "data" / "kol" / "KOL_Posts.csv"
CRM_PATH = BASE_DIR / "data" / "marketing" / "CRM Data.xlsx"
OUT_DIR = BASE_DIR / "_3_marketing"

ADD KOL TO MASTER

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

# -----------------------------
# Helpers
# -----------------------------
def to_num(series: pd.Series) -> pd.Series:
    """Convert strings like '111,112', '0.6%', '  12 ' to numeric floats.
    Percent strings become fractions (0.6% -> 0.006).
    """
    s = series.astype(str).str.strip()
    is_pct = s.str.endswith("%", na=False)
    s = s.str.replace("%", "", regex=False)
    s = s.str.replace(",", "", regex=False)

    out = pd.to_numeric(s, errors="coerce")
    out.loc[is_pct] = out.loc[is_pct] / 100.0
    return out

def parse_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

def pick_metric(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    """Return first existing column converted to numeric."""
    for col in candidates:
        if col in df.columns:
            return to_num(df[col])
    return pd.Series([np.nan] * len(df), index=df.index)

def make_activity_id_kol(out: pd.DataFrame, source_name: str = "KOL") -> pd.Series:
    """Stable-ish id from key fields (uses your kol_* column names)."""
    base = (
        out["kol_restaurant_id"].fillna("").astype(str)
        + "|"
        + out["kol_id"].fillna("").astype(str)
        + "|"
        + out["kol_platform"].fillna("").astype(str)
        + "|"
        + out["kol_post_url"].fillna("").astype(str)
        + "|"
        + out["kol_post_date"].dt.strftime("%Y-%m-%d").fillna("")
    )
    return base.apply(lambda s: f"{source_name}_" + hashlib.md5(s.encode("utf-8")).hexdigest()[:12])

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(KOL_PATH).drop(columns=["Unnamed: 0"], errors="ignore").copy()

# -----------------------------
# Build MASTER (KOL slice)
# -----------------------------
out = pd.DataFrame(index=df.index)

# minimal “master” identifiers
out["source_table"] = "KOL_Posts.csv"
out["channel"] = "KOL"

# KOL core
out["kol_platform"] = df.get("Social Media", np.nan)
out["kol_id"] = df.get("KOL_ID", np.nan)
out["kol_username"] = df.get("Username", np.nan)

# Restaurant keys (still KOL-specific in your design)
out["kol_restaurant_id"] = df.get("Restaurant Code", np.nan)
out["kol_restaurant_name"] = df.get("Restaurant Name", np.nan)
out["kol_country"] = df.get("Country", np.nan)
out["kol_cuisine_type"] = df.get("Cuisine Type", np.nan)

# Post metadata
out["kol_post_url"] = df.get("Post_URL", df.get("Social media URL", np.nan))

# Date: your earlier table didn’t have Post Date; fallback to Booking Date if needed
if "Post Date" in df.columns:
    out["kol_post_date"] = parse_date(df["Post Date"])
else:
    out["kol_post_date"] = parse_date(df.get("Booking Date", pd.Series([np.nan] * len(df), index=df.index)))

# Spend
out["kol_price_per_post"] = to_num(df.get("Price per post", pd.Series([np.nan] * len(df), index=df.index)))
out["kol_cpm"] = to_num(df.get("CPM", pd.Series([np.nan] * len(df), index=df.index)))

# -----------------------------
# KOL metrics (window fallback)
# -----------------------------
out["kol_views"] = pick_metric(df, ["views (latest total)", "Views (+10)", "Views (+5)", "Views"])
out["kol_likes"] = pick_metric(df, ["Likes (+10)", "Likes (+5)", "Likes"])
out["kol_comments"] = pick_metric(df, ["Comments (+10)", "Comments (+5)", "Comments"])

# optional: treat -1 as missing (common encoding)
out.loc[out["kol_likes"] == -1, "kol_likes"] = np.nan
out.loc[out["kol_comments"] == -1, "kol_comments"] = np.nan

out["kol_conv_rate"] = to_num(df.get("Conv Rate", pd.Series([np.nan] * len(df), index=df.index)))
out["kol_growth_rate"] = to_num(df.get("growth rate", pd.Series([np.nan] * len(df), index=df.index)))

# -----------------------------
# Unique activity id
# -----------------------------
out["activity_id"] = make_activity_id_kol(out, source_name="KOL")

# -----------------------------
# Final columns (only what exists)
# -----------------------------
master_cols = [
    "activity_id", "source_table", "channel",
    "kol_platform",
    "kol_id", "kol_username",
    "kol_restaurant_id", "kol_restaurant_name", "kol_country", "kol_cuisine_type",
    "kol_post_url", "kol_post_date",
    "kol_price_per_post", "kol_cpm",
    "kol_views", "kol_likes", "kol_comments",
    "kol_conv_rate", "kol_growth_rate",
]
master_kol = out[master_cols].copy()
master_db = master_kol.copy()

print("MASTER_KOL shape:", master_kol.shape)


MASTER_KOL shape: (923, 19)


,activity_id,source_table,channel,kol_platform,kol_id,kol_username,kol_restaurant_id,kol_restaurant_name,kol_country,kol_cuisine_type,kol_post_url,kol_post_date,kol_price_per_post,kol_cpm,kol_views,kol_likes,kol_comments,kol_conv_rate,kol_growth_rate
0,KOL_78dd89a55819,KOL_Posts.csv,KOL,Instagram,HH-IN36Y,365days2play,1893,Sirimahannop Asiatique The Riverfront,Bangkok,Thai-European,Instagram.com/365days2play,2024-08-23,NaN,0.0,1723.0,NaN,18.0,0.0099,0.006
1,KOL_d342d0c786bc,KOL_Posts.csv,KOL,Instagram,HH-INAAD,aayan_world,4503,Copper Beyond Buffet Gaysorn Amarin (Hungry Hub),Bangkok,International,Instagram.com/aayan_world,2025-09-05,0.0,0.0,30705.0,NaN,47.0,0.0015,109.054
2,KOL_3a536753bc0b,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,NaN,Copper Beyond Buffet Gaysorn Amarin,Bangkok,NaN,Instagram.com/beatricechongyh,2025-08-18,NaN,0.0,111112.0,3572.0,75.0,0.0328,0.584
3,KOL_efe32ed42d46,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,4039,Oishi Grand Siam Paragon,Bangkok,Japanese,Instagram.com/beatricechongyh,2025-08-20,NaN,0.0,102159.0,NaN,70.0,0.0007,0.122
4,KOL_46e9882811d4,KOL_Posts.csv,KOL,Instagram,HH-INBEH,beatricechongyh,5945,Kohaku Omakase,Bangkok,Japanese,Instagram.com/beatricechongyh,2025-08-23,NaN,0.0,102775.0,NaN,44.0,0.0004,0.416
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
918,KOL_d16080ee1c92,KOL_Posts.csv,KOL,Tiktok,NaN,kivalive,NaN,Jim Thompson House,Bangkok,NaN,https://tiktok.com/@kivalive,2025-09-16,NaN,NaN,809.0,NaN,NaN,NaN,-1.000
919,KOL_ff9d0229ca79,KOL_Posts.csv,KOL,NaN,NaN,hunsa.parzing,NaN,Rusty.Bkk,Bangkok,NaN,NaN,2025-11-02,NaN,NaN,NaN,49.0,11.0,NaN,NaN
920,KOL_59d7d64dbcc2,KOL_Posts.csv,KOL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,7.0,0.0,NaN,NaN
921,KOL_59d7d64dbcc2,KOL_Posts.csv,KOL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,3.0,0.0,NaN,NaN


1. BREAKDOWN CRM CAMPAIGN NAME
2. ADD CRM TO MASTER

In [46]:
crm_df = pd.read_excel(CRM_PATH)
print(crm_df.head())
import pandas as pd
import numpy as np

def parse_campaign_name(name):
    if pd.isna(name):
        return pd.Series([np.nan]*11)

    parts = str(name).split("_")

    # Handle non-standard campaigns (e.g. HNY2025)
    if len(parts) < 10:
        return pd.Series([
            np.nan, np.nan, np.nan, np.nan, np.nan,
            np.nan, np.nan, np.nan, np.nan, np.nan,
            name
        ])

    return pd.Series([
        parts[0],                    # lang
        parts[1],                    # city
        parts[2],                    # channel
        parts[3],                    # platform
        parts[4],                    # audience
        parts[5],                    # flag1
        parts[6],                    # flag2
        parts[7],                    # status
        parts[8],                    # date
        parts[9],                    # time
        "_".join(parts[10:])         # topic
    ])

crm_df[[
    "campaign_name_lang",
    "campaign_name_city",
    "campaign_name_channel",
    "campaign_name_platform",
    "campaign_name_audience",
    "campaign_name_flag1",
    "campaign_name_flag2",
    "campaign_name_status",
    "campaign_name_date",
    "campaign_name_time",
    "campaign_name_topic"
]] = crm_df["Campaign Name"].apply(parse_campaign_name)

crm_df["campaign_name_datetime"] = pd.to_datetime(
    crm_df["campaign_name_date"] + crm_df["campaign_name_time"],
    format="%Y%m%d%H%M",
    errors="coerce"
)
import hashlib
import pandas as pd
import numpy as np

def make_activity_id_crm(out: pd.DataFrame, source_name: str = "CRM") -> pd.Series:
    base = (
        out["crm_campaign_id"].fillna("").astype(str)
        + "|"
        + out["crm_sent_datetime"].dt.strftime("%Y-%m-%d %H:%M").fillna("")
        + "|"
        + out["crm_country"].fillna("").astype(str)
    )
    return base.apply(lambda s: f"{source_name}_" + hashlib.md5(s.encode("utf-8")).hexdigest()[:12])


def build_master_crm(crm_df: pd.DataFrame) -> pd.DataFrame:
    df = crm_df.copy()

    # Filter first (so indexes line up)
    df = df.loc[df["Status"].eq("Sent")].copy()

    out = pd.DataFrame(index=df.index)

    # --- Core ---
    out["source_table"] = "CRM_Data.xlsx"
    out["channel"] = "CRM"

    out["crm_campaign_id"] = df.get("Campaign Id", np.nan)
    out["crm_campaign_name"] = df.get("Campaign Name", np.nan)

    out["crm_country"] = df.get("Country", np.nan)
    out["crm_restaurant_id"] = df.get("rest_id", np.nan)

    # Date
    out["crm_sent_datetime"] = parse_date(
        df.get("Sent Date", pd.Series(np.nan, index=df.index))
    )

    # Campaign metadata (from decoded fields)
    out["crm_lang"] = df.get("campaign_name_lang", np.nan)
    out["crm_city"] = df.get("campaign_name_city", np.nan)
    out["crm_audience"] = df.get("campaign_name_audience", np.nan)
    out["crm_topic"] = df.get("campaign_name_topic", np.nan)

    # --- Metrics ---
    out["crm_published"] = to_num(df.get("Published", np.nan))
    out["crm_sent"] = to_num(df.get("Sent", np.nan))
    out["crm_delivered"] = to_num(df.get("Delivered", np.nan))
    out["crm_total_clicked"] = to_num(df.get("Total Clicked", np.nan))
    out["crm_unique_clicked"] = to_num(df.get("Unique Clicked", np.nan))
    out["crm_unique_conversions"] = to_num(df.get("Unique Conversions", np.nan))
    out["crm_revenue"] = to_num(df.get("Revenue", np.nan))

    out["crm_ctr"] = to_num(df.get("CTR", np.nan))
    out["crm_click_to_purchase"] = to_num(df.get("Click_to_purchase", np.nan))
    out["crm_delivered_to_purchase"] = to_num(df.get("delivered_to_purchase", np.nan))
    out["crm_rev_per_click"] = to_num(df.get("Rev_per_click", np.nan))

    # --- Unique ID ---
    out["activity_id"] = make_activity_id_crm(out)

    master_cols = [
        "activity_id", "source_table", "channel",
        "crm_campaign_id", "crm_campaign_name",
        "crm_country", "crm_restaurant_id",
        "crm_sent_datetime",
        "crm_lang", "crm_city", 
        "crm_audience", "crm_topic",
        "crm_published", "crm_sent", "crm_delivered",
        "crm_total_clicked", "crm_unique_clicked",
        "crm_unique_conversions", "crm_revenue",
        "crm_ctr", "crm_click_to_purchase",
        "crm_delivered_to_purchase", "crm_rev_per_click",
    ]

    return out[[c for c in master_cols if c in out.columns]].copy()

master_crm = build_master_crm(crm_df)
master_crm


   Campaign Id                                      Campaign Name Status  \
0       1350.0  TH_BKK_ctnoti_netcore_group_N_N_active_2024123...   Sent   
1       1351.0  EN_BKK_ctnoti_netcore_group_N_N_active_2024123...   Sent   
2       1353.0  EN_BKK_ctnoti_netcore_group_N_N_active_2024123...   Sent   
3       1352.0  TH_BKK_ctnoti_netcore_group_N_N_active_2024123...   Sent   
4       1355.0  EN_BKK_ctnoti_netcore_group_N_N_active_2024123...   Sent   

                           Subject line or Title Desc.           Sent Date  \
0       🔥 ร้านในห้างเด็ด ที่ต้องไปก่อนเคาท์ดาวน์   NaN 2024-12-31 11:13:00   
1  🔥 Must-Try Mall Restaurants Before Countdown!   NaN 2024-12-31 11:15:00   
2                🍣 Oishi Grand – Year-End Feast!   NaN 2024-12-31 14:05:00   
3                   🍣 โออิชิแกรนด์ คุ้มส่งท้ายปี   NaN 2024-12-31 14:05:00   
4                       💥 Just a Few Hours Left!   NaN 2024-12-31 16:00:00   

   Published     Sent  Delivered  Total Clicked  ...  is under  \
0    539

,activity_id,source_table,channel,crm_campaign_id,crm_campaign_name,crm_country,crm_restaurant_id,crm_sent_datetime,crm_lang,crm_city,...,crm_sent,crm_delivered,crm_total_clicked,crm_unique_clicked,crm_unique_conversions,crm_revenue,crm_ctr,crm_click_to_purchase,crm_delivered_to_purchase,crm_rev_per_click
0,CRM_ebb78f5ff63c,CRM_Data.xlsx,CRM,1350.0,TH_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,2024-12-31 11:13:00,TH,BKK,...,53947.0,45537.0,254.0,247.0,11.0,32800.0,0.0054,0.0445,0.000242,132.793522
1,CRM_1a230631d9f0,CRM_Data.xlsx,CRM,1351.0,EN_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,2024-12-31 11:15:00,EN,BKK,...,42787.0,34220.0,302.0,284.0,5.0,15773.0,0.0083,0.0176,0.000146,55.538732
2,CRM_faa734b54a04,CRM_Data.xlsx,CRM,1353.0,EN_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,2024-12-31 14:05:00,EN,BKK,...,42819.0,6415.0,120.0,120.0,0.0,0.0,0.0187,0.0000,0.000000,0.000000
3,CRM_e98ac083b223,CRM_Data.xlsx,CRM,1352.0,TH_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,2024-12-31 14:05:00,TH,BKK,...,54066.0,12199.0,134.0,134.0,2.0,1221.0,0.0110,0.0149,0.000164,9.111940
4,CRM_fff0fd219bbd,CRM_Data.xlsx,CRM,1355.0,EN_BKK_ctnoti_netcore_group_N_N_active_2024123...,Thailand,NaN,2024-12-31 16:00:00,EN,BKK,...,42866.0,34583.0,311.0,303.0,4.0,14525.0,0.0088,0.0132,0.000116,47.937294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2349,CRM_e6dcd7bb7d78,CRM_Data.xlsx,CRM,4293.0,TH_BKK_ctnoti_netcore_group_N_N_active_2026011...,Thailand,NaN,2026-01-12 15:31:00,TH,BKK,...,77434.0,63910.0,209.0,205.0,4.0,6642.0,0.0032,0.0195,0.000063,32.400000
2350,CRM_0aa0bf8b7c0e,CRM_Data.xlsx,CRM,4294.0,EN_BKK_ctnoti_netcore_group_N_N_active_2026011...,Thailand,NaN,2026-01-12 15:34:00,EN,BKK,...,51469.0,42007.0,154.0,145.0,1.0,4464.0,0.0035,0.0069,0.000024,30.786207
2352,CRM_aa16e95c6560,CRM_Data.xlsx,CRM,4296.0,EN_BKK_ctnoti_netcore_group_N_N_active_2026011...,Thailand,NaN,2026-01-12 17:15:00,EN,BKK,...,51378.0,42175.0,123.0,116.0,0.0,0.0,0.0028,0.0000,0.000000,0.000000
2353,CRM_633251ea2c2e,CRM_Data.xlsx,CRM,4297.0,TH_BKK_ctnoti_netcore_single_N_N_active_202601...,Thailand,NaN,2026-01-12 19:00:00,TH,BKK,...,77256.0,63907.0,178.0,170.0,3.0,5780.0,0.0027,0.0176,0.000047,34.000000
